In [1]:
import sys
import pandas as pd

sys.path.append("../src")  # permet d'importer les fichiers de src/
from metrics import GROUPES, evaluer, rapport_par_groupe, probas_vers_predictions, chercher_seuils
train = pd.read_csv("../data/processed/train.csv", dtype={"id": str})
val = pd.read_csv("../data/processed/val.csv", dtype={"id": str})

print("Train :", train.shape, "| Validation :", val.shape)
print("Import de metrics.py : OK")

Train : (13993, 11) | Validation : (3006, 11)
Import de metrics.py : OK


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectoriseur = TfidfVectorizer(
    lowercase=True,         # tout en minuscules
    stop_words="english",   # retire les mots vides : the, of, and...
    ngram_range=(1, 2),     # mots seuls ET paires de mots ("neural network")
    min_df=2,               # ignore les mots présents dans moins de 2 résumés
    max_features=100_000,   # garde au maximum 100 000 mots/paires
    sublinear_tf=True,      # atténue l'effet d'un mot répété 10 fois
)

X_train = vectoriseur.fit_transform(train["text"])  # apprend le vocabulaire SUR LE TRAIN uniquement
X_val = vectoriseur.transform(val["text"])          # réutilise ce vocabulaire sur la validation

print("X_train :", X_train.shape)
print("X_val   :", X_val.shape)

X_train : (13993, 100000)
X_val   : (3006, 100000)


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

y_train = train[GROUPES].values  # les 8 colonnes de 0/1
y_val = val[GROUPES].values

modele = OneVsRestClassifier(LogisticRegression(max_iter=1000, C=4.0))
modele.fit(X_train, y_train)
print("Entraînement terminé")

Entraînement terminé


In [4]:
probas_val = modele.predict_proba(X_val)        # probabilités entre 0 et 1, forme (2609, 8)
pred_val = probas_vers_predictions(probas_val)  # seuil 0.5 -> 0 ou 1

scores = evaluer(y_val, pred_val)
for nom, valeur in scores.items():
    print(f"{nom:16s} {valeur:.4f}")
print()
print(rapport_par_groupe(y_val, pred_val))

micro_f1         0.8581
macro_f1         0.5116
hamming_loss     0.0401
subset_accuracy  0.7688

              precision    recall  f1-score   support

          cs       0.88      0.89      0.89       973
        math       0.90      0.80      0.85       717
     physics       0.96      0.92      0.94      1497
       q-bio       1.00      0.14      0.24        58
       q-fin       0.90      0.16      0.28        55
        stat       0.82      0.31      0.45       146
        eess       0.66      0.23      0.34       125
        econ       0.60      0.06      0.11        52

   micro avg       0.92      0.81      0.86      3623
   macro avg       0.84      0.44      0.51      3623
weighted avg       0.91      0.81      0.84      3623
 samples avg       0.90      0.86      0.87      3623



In [5]:
import json

with open("../results/baseline_tfidf_val.json", "w") as f:
    json.dump(scores, f, indent=2)
print("Scores sauvegardés dans results/baseline_tfidf_val.json")

Scores sauvegardés dans results/baseline_tfidf_val.json


In [6]:
seuils = chercher_seuils(y_val, probas_val)
print("Seuils choisis :", dict(zip(GROUPES, seuils.tolist())))
print()

pred_val_seuils = probas_vers_predictions(probas_val, seuils)
scores_seuils = evaluer(y_val, pred_val_seuils)
for nom, valeur in scores_seuils.items():
    print(f"{nom:16s} {valeur:.4f}")
print()
print(rapport_par_groupe(y_val, pred_val_seuils))

with open("../results/baseline_tfidf_val_seuils.json", "w") as f:
    json.dump({"seuils": seuils.tolist(), **scores_seuils}, f, indent=2)
print("Scores sauvegardés dans results/baseline_tfidf_val_seuils.json")

Seuils choisis : {'cs': 0.45, 'math': 0.4, 'physics': 0.45, 'q-bio': 0.15, 'q-fin': 0.1, 'stat': 0.2, 'eess': 0.15, 'econ': 0.1}

micro_f1         0.8672
macro_f1         0.7008
hamming_loss     0.0403
subset_accuracy  0.7645

              precision    recall  f1-score   support

          cs       0.88      0.91      0.89       973
        math       0.86      0.84      0.85       717
     physics       0.96      0.93      0.95      1497
       q-bio       0.76      0.45      0.57        58
       q-fin       0.57      0.60      0.58        55
        stat       0.61      0.62      0.61       146
        eess       0.46      0.71      0.56       125
        econ       0.55      0.63      0.59        52

   micro avg       0.86      0.87      0.87      3623
   macro avg       0.71      0.71      0.70      3623
weighted avg       0.87      0.87      0.87      3623
 samples avg       0.90      0.91      0.89      3623

Scores sauvegardés dans results/baseline_tfidf_val_seuils.json
